<h1>Importing</h1>

In [1]:
import pandas as pd
import numpy as np
import os

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from concurrent.futures import ThreadPoolExecutor, as_completed

c:\Users\admin\OneDrive\Desktop\AirBNB project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h1>setting up </h1>

In [24]:
df = pd.read_parquet('./dataset/dataset_temp.parquet')

In [3]:
df

,Price,description,city,guests,bedrooms,beds,baths,luxury_items,s3_url,sentence
0,102699.0,Situated in one of Seattle's friendliest and f...,Seattle,10.0,5.0,5.0,3.5,0.0,http://localhost:9000/airbnb-images/image_1.jpg,This image consists of these items with highes...
1,208528.0,Midcentury modern 2bed/2bath stilt home with s...,Los Angeles,6.0,2.0,3.0,2.0,0.0,http://localhost:9000/airbnb-images/image_2.jpg,This image consists of these items with highes...
2,16264.0,Welcome to UNTITLED at 3 Freeman Alley!Our Stu...,New York,2.0,1.0,1.0,0.0,1.0,http://localhost:9000/airbnb-images/image_3.jpg,This image consists of these items with highes...
3,70478.0,Kick back in this peaceful room with an en-sui...,San Francisco,2.0,1.0,1.0,1.5,0.0,http://localhost:9000/airbnb-images/image_5.jpg,This image consists of these items with highes...
4,157811.0,Step into serenity at this beautifully curated...,Burbank,6.0,3.0,3.0,2.0,0.0,http://localhost:9000/airbnb-images/image_6.jpg,This image consists of these items with highes...
...,...,...,...,...,...,...,...,...,...,...
1699,120658.0,This spacious duplex gem in the heart of Houst...,Houston,6.0,3.0,4.0,2.0,0.0,http://localhost:9000/airbnb-images/image_2106...,This image consists of these items with highes...
1700,66435.0,"This special place is close to everything, loc...",Boston,2.0,0.0,1.0,1.0,1.0,http://localhost:9000/airbnb-images/image_2108...,This image consists of these items with highes...
1701,95358.0,Moody Blues in Music City is newly furnished!S...,Nashville,6.0,1.0,4.0,1.0,0.0,http://localhost:9000/airbnb-images/image_2109...,This image consists of these items with highes...
1702,143942.0,"Explore a beautifully reimagined luxury hotel,...",Boston,2.0,1.0,1.0,0.0,1.0,http://localhost:9000/airbnb-images/image_2110...,This image consists of these items with highes...


<h1>NN prompting</h1>

In [4]:
TEMPLATE = ChatPromptTemplate.from_messages([
  ('system', """You are a real estate analyst. Given an Airbnb listing description (which can be truncated) and image tags, output exactly three values separated by "|":
- luxury_score (0-100) – A numeric value desribing how luxurious/high‑end the listing is.
- location_desirability (0-100) –  A numeric value desribing how desirable of a getaway this listing is.
- quality – one of: poor, acceptable, great, perfect.

Example: 45 | 70 | great
Output only the three values, nothing else. No explanations. Adhere strictly to the output format, requested by the user."""),
  ('human', """
You are given one listing. This listing adheres to this format:
Description | Image description

Here is the listing:
{listings_list}

Output exactly one line with this format:
luxury: <0-100> | location: <0-100> | quality: <poor/acceptable/great/perfect>
   """)])

<h2>Сначала скачать Ollama -> ollama pull mistral:instruct -> ollama run mistral:instruct</h2>

In [5]:
lst = []
for i in df.index:
    desc = df.loc[i, 'description'][:1000].replace('\n', ' ').replace('|', ';')
    img_desc = df.loc[i, 'sentence'].replace('\n', ' ')
    lst.append(f'{desc} | {img_desc}')

def NN_dialogue(text: str) -> str:

  llm = ChatOllama(
    model = 'mistral:instruct',
    temperature=0.1,
    num_predict=1024
  )

  
  prompt = TEMPLATE.format_messages(listings_list = text)
  
  response = llm.invoke(prompt)
  
  return response.content
  

In [7]:
def send_request(i) -> str:
  QUERY = lst[i]
  answer = NN_dialogue(QUERY)
  return '\n'.join(answer.split('\n'))

In [8]:
temp = []

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(send_request, idx): idx for idx in df.index}
    for future in as_completed(futures):
        result = future.result()
        temp.append((futures[future], result))

In [16]:
temp = sorted(temp, key=lambda x: x[0])
df_list = []
for tup in temp:
  st = tup[1]
  st = st.replace('luxury:', '').replace('location:', '').replace('quality:', '').replace(' ', '')
  df_list.append(tuple(st.split('|')))


In [19]:
new_features = pd.DataFrame(df_list, columns=['luxury_score', 'location_desirability', 'quality'])

In [25]:
end_df = pd.concat([df, new_features], axis=1).drop(columns=['s3_url', 'sentence'])
end_df

,Price,description,city,guests,bedrooms,beds,baths,luxury_items,luxury_score,location_desirability,quality
0,102699.0,Situated in one of Seattle's friendliest and f...,Seattle,10.0,5.0,5.0,3.5,0.0,85,90,great
1,208528.0,Midcentury modern 2bed/2bath stilt home with s...,Los Angeles,6.0,2.0,3.0,2.0,0.0,90,95,great
2,16264.0,Welcome to UNTITLED at 3 Freeman Alley!Our Stu...,New York,2.0,1.0,1.0,0.0,1.0,50,80,great
3,70478.0,Kick back in this peaceful room with an en-sui...,San Francisco,2.0,1.0,1.0,1.5,0.0,85,80,great
4,157811.0,Step into serenity at this beautifully curated...,Burbank,6.0,3.0,3.0,2.0,0.0,75,80,great
...,...,...,...,...,...,...,...,...,...,...,...
1699,120658.0,This spacious duplex gem in the heart of Houst...,Houston,6.0,3.0,4.0,2.0,0.0,65,85,great
1700,66435.0,"This special place is close to everything, loc...",Boston,2.0,0.0,1.0,1.0,1.0,50,85,great
1701,95358.0,Moody Blues in Music City is newly furnished!S...,Nashville,6.0,1.0,4.0,1.0,0.0,90,85,great
1702,143942.0,"Explore a beautifully reimagined luxury hotel,...",Boston,2.0,1.0,1.0,0.0,1.0,90,85,great


In [29]:
end_df['luxury_score'] = pd.to_numeric(end_df['luxury_score'])
end_df['location_desirability'] = pd.to_numeric(end_df['location_desirability'])
end_df.to_parquet('./dataset/forCatboost.parquet')